In [1]:
from ingest import load_faq_data
documents= load_faq_data()

In [2]:
documents[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [3]:
documents_llm =[]
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)
        
len(documents_llm)   

161

In [4]:
documents = documents_llm


In [5]:
doc =documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions : list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()


In [8]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()


In [9]:
import json

user_prompt = json.dumps(doc)


In [10]:
messages = [
    {"role": "assistant", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]


In [11]:
response = client.messages.parse(
    model="claude-haiku-5-5",
    max_tokens=1024,
    system=data_gen_instructions,
    messages=[{"role": "user", "content": user_prompt}],
    output_format=Questions,
)

results = response.parsed_output.questions
results


['I just found out about this LLM Zoomcamp course, is it still possible to join now that it has already started?',
 'Hey, I only discovered the course recently. Am I still able to enroll and take part in it?',
 'If I sign up for llm-zoomcamp late, can I still get a certificate at the end?',
 "I stumbled upon the LLM Zoomcamp today, can I still jump in even though I'm late, and will I be able to get a certificate if I submit my project on time?",
 'Is it too late to start the LLM Zoomcamp if I only just heard about it, and do I need to hand in my project before the deadline to earn a certificate?']

In [12]:
from evaluation_utils import llm_structured

In [13]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured(
        client=client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
        model="claude-haiku-5-5",
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "course": doc["course"],
            "document": doc["id"],
        })

    return results, usage


In [14]:
generate_ground_truth(doc)

([{'question': "Hey, I only just found out about the LLM Zoomcamp. Is it still possible to join even though it's already started?",
   'course': 'llm-zoomcamp',
   'document': '74eb249bbf'},
  {'question': "I'm thinking of joining the llm-zoomcamp late. Can I still get a certificate at the end if I do?",
   'course': 'llm-zoomcamp',
   'document': '74eb249bbf'},
  {'question': 'Is there a deadline for submitting the project if I want a certificate? I just discovered the course recently.',
   'course': 'llm-zoomcamp',
   'document': '74eb249bbf'},
  {'question': "I've only just come across this course and I'm wondering whether it's too late to enroll in LLM Zoomcamp.",
   'course': 'llm-zoomcamp',
   'document': '74eb249bbf'},
  {'question': 'If I start the course late, do I still have to submit my project before submissions close to get a certificate?',
   'course': 'llm-zoomcamp',
   'document': '74eb249bbf'}],
 Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral

In [15]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [16]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [17]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/161 [00:00<?, ?it/s]

In [18]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

804

In [19]:
ground_truth[10]

{'question': 'Where can I find the zoom link for the office hours or live workshop sessions in llm-zoomcamp?',
 'course': 'llm-zoomcamp',
 'document': '489dd1c9d9'}

In [20]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.0318753

In [21]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.0318753

In [22]:
import pandas as pd
df_ground_truth = pd.DataFrame(ground_truth)

In [23]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

OSError: Cannot save file into a non-existent directory: 'data'

In [24]:
len(df_ground_truth)

804

In [27]:
result, usage = llm_structured(
        client=client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
        model="claude-haiku-5-5",
    )

print(result.questions)

['I just found out about this LLM Zoomcamp course, is it still possible to join it now even though it has already started?', 'Hey, I only just discovered the course. Am I still able to enroll and take part in it?', 'If I join the LLM Zoomcamp late, can I still get a certificate at the end?', 'Is it too late to sign up for llm-zoomcamp? I just heard about it and want to know if I can still participate.', 'I found the LLM Zoomcamp recently. Can I still join the course now, and do I need to do anything special to get a certificate?']


In [28]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records


[{'question': 'I just found out about this LLM Zoomcamp course, is it still possible to join it now even though it has already started?',
  'document': 'c2903069a0'},
 {'question': 'Hey, I only just discovered the course. Am I still able to enroll and take part in it?',
  'document': 'c2903069a0'},
 {'question': 'If I join the LLM Zoomcamp late, can I still get a certificate at the end?',
  'document': 'c2903069a0'},
 {'question': 'Is it too late to sign up for llm-zoomcamp? I just heard about it and want to know if I can still participate.',
  'document': 'c2903069a0'},
 {'question': 'I found the LLM Zoomcamp recently. Can I still join the course now, and do I need to do anything special to get a certificate?',
  'document': 'c2903069a0'}]

In [29]:
from evaluation_utils import llm_structured_retry


In [30]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        client=client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
        model="claude-haiku-5-5",
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage


In [31]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)


  0%|          | 0/5 [00:00<?, ?it/s]

In [32]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress


In [33]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)


804

In [34]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost


0.0318753

In [35]:
from evaluation_utils import calc_total_price

calc_total_price(usages)


0.0318753

In [36]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)


In [37]:
df_ground_truth.to_csv("ground_truth-new.csv", index=False)
